# Quantum Kernel-Based Classifier


1. **Quantum feature encoding**: $x \;\longrightarrow\; |\psi(x)\rangle = U(x)\,|0\rangle^{\otimes n}$
    - $x \in \mathbb{R}^d$ = classical data

    - $U(x)$ = quantum feature map

	- $|\psi(x)\rangle \in \mathbb{C}^{2^n}$ = quantum state in Hilbert space

2. **Quantum Fidelity Kernel**: $K(x_i, x_j) = \left|\langle \psi(x_i) \mid \psi(x_j) \rangle\right|^2 = \left|\langle 0 |U^\dagger(x_j)\,U(x_i)|0\rangle\right|^2$

3. **Kernel Matrix Construction**: For dataset $\{x_1, \dots, x_N\}:$, $K_{ij} = K(x_i, x_j)$, and the Kernel Matrix becomes $K = \begin{bmatrix}
K(x_1,x_1) & \cdots & K(x_1,x_N) \\ \vdots & \ddots & \vdots \\ K(x_N,x_1) & \cdots & K(x_N,x_N) \end{bmatrix}$

4. **SVM Decision Function**: Using the Quantum Kernel, the Classical SVM function becomes $f(x) = \sum_{i \in SV} \alpha_i \, y_i \, K(x_i, x) + b$
    - $SV$ = support vectors
	- $\alpha_i$ = learned classical weights
	- $y_i \in \{-1, +1\}$ = labels
	- $b$ = bias

Thus prediction becomes, $\hat{y} = \operatorname{sign}(f(x))$

#### Advantages of Quantum Kernel Methods

1. **Richer feature spaces**: Quantum feature maps can embed data into very high‑dimensional Hilbert spaces, which may make classes more separable.

2. **Implicit kernels**: Use Kernels that are hard to compute classically but are natural to estimate as state overlaps on a quantum device.

3. **Near‑term friendly**: Kernel evaluation circuits are often shallower than full variational training, which can be more robust on NISQ hardware.

4. **Classical training**: Once the kernel matrix is computed, you can use well‑understood classical SVM solvers.

In [12]:
! pip3 install qiskit-machine-learning qiskit-algorithms numpy matplotlib scikit-learn --break-system-packages

In [13]:
import numpy as np
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.circuit.library import ZZFeatureMap
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons

#### Step 1: Custom Fidelity Function for Quantum Kernel

Defines a custom function to compute fidelity (state overlap) using `StatevectorSampler`, which gives a custom, explicit way to calculate kernel values from Quantum states.

In [14]:
def custom_fidelity(sampler, circuit_1, circuit_2):
    combined_circuit = circuit_1.compose(circuit_2.inverse())  # Combine the circuits
    job = sampler.run([combined_circuit])
    result = job.result()
    probabilities = result.quasi_dists[0]

    # Fidelity is the probability of measuring the all-zero state
    fidelity = probabilities.get(0, 0)
    return fidelity

#### Step 2: Generate Synthetic Dataset (Two-Moons Dataset)

Creates a non‑linearly separable dataset and splits into train/test, a classic benchmark dataset to show the benefit of non-linear kernels.

In [15]:
X, y = make_moons(n_samples=100, noise=0.1, random_state=42)

# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

#### Step 3: Define Quantum Feature Map

Builds a feature map circuit that encodes classical inputs into quantum states, in which defines the quantum feature space used by the kernel.

In [16]:
feature_map = ZZFeatureMap(
    feature_dimension=2,
    reps=2,
    entanglement='linear'
)

/var/folders/td/rtvtvvjn3x1gfqswnc7bb6hm0000gn/T/ipykernel_57593/2279381982.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


#### Step 4: Initialize the Updated Sampler

Creates a `StatevectorSampler` for circuit evaluation to provide exact state overlaps in simulation (no sampling noise).

In [17]:
sampler = StatevectorSampler()

#### Step 5: Define Fidelity Quantum Kernel with Custom Fidelity Function

Used to wrap the quantum overlap computation into a kernel object usable by the SVM.

In [18]:
class CustomFidelityQuantumKernel(FidelityQuantumKernel):

    def __init__(self, feature_map, sampler):
        super().__init__(feature_map=feature_map, fidelity=None)
        self.sampler = sampler

    def _compute_fidelity(self, circuit_1, circuit_2):
        return custom_fidelity(self.sampler, circuit_1, circuit_2)

#### Step 6: Compute the Quantum Kernel Matrix

Evaluates $K(x_i, x_j)$ for all pairs in the training set, and produces the kernel matrix needed by the SVM.

In [19]:
quantum_kernel = CustomFidelityQuantumKernel(
    feature_map=feature_map,
    sampler=sampler
)

X_train_kernel = quantum_kernel.evaluate(X_train)
X_test_kernel = quantum_kernel.evaluate(X_test, X_train)

#### Step 7: Train a Classical Support Vector Machine (SVM) with the Quantum Kernel

Fits the SVM using the precomputed quantum kernel matrix. The learning happens classically but the SVM captures the quantum similarity.

In [20]:
svm = SVC(kernel="precomputed")
svm.fit(X_train_kernel, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'precomputed'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


#### Step 8: Evaluate Model Accuracy

In [21]:
y_pred = svm.predict(X_test_kernel)

accuracy = np.mean(y_pred == y_test)
print(f"Quantum Kernel SVM Accuracy: {accuracy * 100:.2f}%")

Quantum Kernel SVM Accuracy: 60.00%
